<a href="https://colab.research.google.com/github/Alister44/DTA_2026_DE/blob/main/Unterrichtsaufgaben/demo_stat_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Statistische Tests in Python — Demonstration

**Dauer:** ~25 Minuten · Bibliothek `scipy.stats`

---

Dies ist ein Demonstrations-Notebook zur Vorlesung über p-Werte. Wir werden sehen, dass technisch gesehen jeder Test **nur wenige Zeilen Code** ist. Die ganze Schwierigkeit liegt darin zu verstehen, *welchen* Test man wählt und *wie* man das Ergebnis liest.

**Erinnerung aus der Vorlesung:**
- `p < 0.05` → Ergebnis ist **statistisch signifikant** → wir lehnen H₀ ab (es gibt einen Effekt)
- `p ≥ 0.05` → **nicht signifikant** → wir lehnen H₀ nicht ab (kein Beweis für einen Effekt)

> 💡 Alle Tests geben zwei Hauptwerte zurück: **die Teststatistik** und den **p-Wert**. Uns interessiert fast immer der p-Wert.



## Vorbereitung


In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("https://raw.githubusercontent.com/Alister44/DTA_2026_DE/refs/heads/main/data/shop_customers.csv")
print("Größe:", df.shape)
df.head()


Größe: (500, 11)


,customer_id,gender,age,country,channel,device,spend,session_min,sat_before,sat_after,purchased
0,1001,m,23,Germany,Advertising,Desktop,997.11,22.2,6,7,Yes
1,1002,m,38,Ukraine,Organic,Desktop,510.09,19.8,5,8,No
2,1003,m,20,Ukraine,Social Media,Desktop,789.71,21.0,7,8,No
3,1004,m,40,Germany,Social Media,Desktop,1041.02,17.4,8,9,No
4,1005,m,18,Ukraine,Advertising,Desktop,944.34,23.5,6,7,No


---
## Test 1. Einstichproben-t-Test

**Frage:** Unterscheiden sich die durchschnittlichen Ausgaben der Kunden von den erwarteten 900 UAH?

**Hypothesen:**
- H₀: durchschnittliche Ausgaben = 900
- H₁: durchschnittliche Ausgaben ≠ 900

**Funktion:** `stats.ttest_1samp(Daten, erwarteter_Wert)`


In [2]:
mean_spend = df["spend"].mean()
print(f"Tatsächlicher Mittelwert: {mean_spend:.1f} UAH")

t_stat, p_value = stats.ttest_1samp(df["spend"], 900)
print(f"t-Statistik: {t_stat:.3f}")
print(f"p-Wert:      {p_value:.4f}")

if p_value < 0.05:
    print("\n✅ p < 0.05 → signifikant: der Mittelwert UNTERSCHEIDET sich von 900")
else:
    print("\n❌ p ≥ 0.05 → nicht signifikant: kein Grund zu sagen, dass der Mittelwert ≠ 900 ist")


Tatsächlicher Mittelwert: 845.0 UAH
t-Statistik: -3.687
p-Wert:      0.0003

✅ p < 0.05 → signifikant: der Mittelwert UNTERSCHEIDET sich von 900


**Interpretation:** p ≈ 0.0003 < 0.05 → wir lehnen H₀ ab. Die durchschnittlichen Ausgaben (≈845 UAH) unterscheiden sich statistisch signifikant von 900 UAH. Der Unterschied ist nicht zufällig.


---
## Test 2. Zweistichproben-t-Test (unabhängige Gruppen)

**Frage:** Unterscheiden sich die Ausgaben von Männern und Frauen?

**Hypothesen:**
- H₀: durchschnittliche Ausgaben Männer = Frauen
- H₁: sie unterscheiden sich

**Funktion:** `stats.ttest_ind(Gruppe1, Gruppe2)`


In [3]:
spend_m = df[df["gender"] == "m"]["spend"]
spend_f = df[df["gender"] == "f"]["spend"]

print(f"Männer: {spend_m.mean():.1f} UAH")
print(f"Frauen: {spend_f.mean():.1f} UAH")

t_stat, p_value = stats.ttest_ind(spend_m, spend_f)
print(f"\np-Wert: {p_value:.4f}")

if p_value < 0.05:
    print("✅ Signifikanter Unterschied zwischen den Geschlechtern")
else:
    print("❌ Kein signifikanter Unterschied festgestellt")


Männer: 848.7 UAH
Frauen: 841.3 UAH

p-Wert: 0.8038
❌ Kein signifikanter Unterschied festgestellt


**Interpretation:** p ≈ 0.80 > 0.05 → wir **lehnen H₀ nicht ab**. Wir haben keinen signifikanten Unterschied in den Ausgaben zwischen Männern und Frauen festgestellt. Beachten Sie: Wir sagen „nicht festgestellt“, nicht „bewiesen, dass es keinen Unterschied gibt“!


---
## Test 3. Gepaarter t-Test (vorher und nachher)

**Frage:** Ist die Kundenzufriedenheit nach dem Redesign der Website gestiegen?

Hier vergleichen wir ZWEI Messungen bei DENSELBEN Kunden (vorher und nachher) — deshalb ist der Test **gepaart**.

**Funktion:** `stats.ttest_rel(vorher, nachher)`


In [4]:
print(f"Zufriedenheit VORHER:  {df['sat_before'].mean():.2f}")
print(f"Zufriedenheit NACHHER: {df['sat_after'].mean():.2f}")

t_stat, p_value = stats.ttest_rel(df["sat_before"], df["sat_after"])
print(f"\np-Wert: {p_value:.6f}")

if p_value < 0.05:
    print("✅ Die Zufriedenheit hat sich signifikant verändert")
else:
    print("❌ Keine signifikante Veränderung festgestellt")


Zufriedenheit VORHER:  6.54
Zufriedenheit NACHHER: 7.14

p-Wert: 0.000000
✅ Die Zufriedenheit hat sich signifikant verändert


**Interpretation:** p < 0.001 → wir lehnen H₀ ab. Die Zufriedenheit ist signifikant gestiegen (von 6.54 auf 7.14). Das Redesign hat funktioniert.

> 💡 Warum gerade ein gepaarter Test? Weil jedes „nachher“ mit seinem eigenen „vorher“ verknüpft ist (derselbe Kunde). Der gepaarte Test berücksichtigt diesen Zusammenhang und ist deshalb aussagekräftiger.


---
## Test 4. ANOVA (Vergleich von 3+ Gruppen)

**Frage:** Unterscheiden sich die Ausgaben je nach Akquisitionskanal (Organic / Advertising / Social Media)?

Es gibt **drei** Gruppen, daher ist ein t-Test nicht geeignet — wir verwenden **ANOVA**.

**Funktion:** `stats.f_oneway(Gruppe1, Gruppe2, Gruppe3)`


In [5]:
for channel in df["channel"].unique():
    avg = df[df["channel"] == channel]["spend"].mean()
    print(f"{channel}: {avg:.1f} UAH")

groups = [df[df["channel"] == ch]["spend"] for ch in df["channel"].unique()]

f_stat, p_value = stats.f_oneway(*groups)
print(f"\nF-Statistik: {f_stat:.2f}")
print(f"p-Wert:      {p_value:.6f}")

if p_value < 0.05:
    print("✅ Mindestens ein Kanal unterscheidet sich signifikant")
else:
    print("❌ Kein signifikanter Unterschied zwischen den Kanälen")


Advertising: 1057.9 UAH
Organic: 791.3 UAH
Social Media: 698.4 UAH

F-Statistik: 63.54
p-Wert:      0.000000
✅ Mindestens ein Kanal unterscheidet sich signifikant


**Interpretation:** p < 0.001 → wir lehnen H₀ ab. Die Kanäle unterscheiden sich signifikant in den Ausgaben (Advertising ≈1058 UAH liegt deutlich höher als Social Media ≈698).

> ⚠️ ANOVA sagt nur, dass *irgendein* Unterschied besteht, aber nicht, *zwischen welchen* Kanälen genau. Um das herauszufinden, braucht man zusätzliche (Post-hoc-)Tests — das würde den Rahmen der heutigen Demonstration sprengen.


---
## Test 5. Chi-Quadrat-Test (Zusammenhang von Kategorien)

**Frage:** Hängt der Gerätetyp (Mobile/Desktop) mit dem Kauf zusammen (Ja/Nein)?

Beide Variablen sind **kategorial** → Chi-Quadrat.

**Schritt 1:** Wir erstellen eine Kontingenztabelle. **Schritt 2:** `stats.chi2_contingency(Tabelle)`


In [6]:
contingency = pd.crosstab(df["device"], df["purchased"])
print("Kontingenztabelle:")
print(contingency)

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"\nchi2: {chi2:.3f}")
print(f"p-Wert: {p_value:.4f}")

if p_value < 0.05:
    print("✅ Der Gerätetyp HÄNGT mit dem Kauf zusammen")
else:
    print("❌ Kein Zusammenhang festgestellt")


Kontingenztabelle:
purchased   No  Yes
device             
Desktop    110   79
Mobile     223   88

chi2: 9.039
p-Wert: 0.0026
✅ Der Gerätetyp HÄNGT mit dem Kauf zusammen


**Interpretation:** p ≈ 0.003 < 0.05 → wir lehnen H₀ ab. Der Gerätetyp hängt mit der Kaufwahrscheinlichkeit zusammen (auf Desktop wird häufiger gekauft). Das ist ein nützlicher Insight für das Business!


---
## Test 6. Korrelation (Zusammenhang zweier Zahlen)

**Frage:** Hängt die Zeit auf der Website mit der Ausgabensumme zusammen?

Beide Variablen sind **numerisch** → Pearson-Korrelation.

**Funktion:** `stats.pearsonr(Variable1, Variable2)` — gibt den Koeffizienten r und den p-Wert zurück.


In [7]:
r, p_value = stats.pearsonr(df["session_min"], df["spend"])
print(f"Korrelationskoeffizient r: {r:.3f}")
print(f"p-Wert: {p_value:.6f}")

# r nahe 1 → starker positiver Zusammenhang; nahe 0 → kein Zusammenhang
if p_value < 0.05:
    print("✅ Der Zusammenhang ist statistisch signifikant")
else:
    print("❌ Kein signifikanter Zusammenhang")


Korrelationskoeffizient r: 0.732
p-Wert: 0.000000
✅ Der Zusammenhang ist statistisch signifikant


**Interpretation:** r ≈ 0.73 (starker positiver Zusammenhang) und p < 0.001 (signifikant). Je mehr Zeit ein Kunde auf der Website verbringt, desto mehr gibt er aus.

> ⚠️ **Korrelation ≠ Kausalität!** Möglicherweise verursacht die Zeit die Ausgaben, möglicherweise ist es umgekehrt, oder es gibt einen dritten Faktor (Interesse). Der Test sagt nur, dass der Zusammenhang real ist.


---
# Zusammenfassung der Demonstration

Wir haben **6 verschiedene Tests** durchgeführt, und jeder ist buchstäblich 1-2 Zeilen Code:

| Test | scipy-Funktion | Wann anwenden |
|------|---------------|-------------------|
| Einstichproben-t | `ttest_1samp` | Mittelwert gegen eine feste Zahl |
| Zweistichproben-t | `ttest_ind` | 2 unabhängige Gruppen vergleichen |
| Gepaarter t | `ttest_rel` | vorher/nachher bei denselben Objekten |
| ANOVA | `f_oneway` | 3+ Gruppen vergleichen |
| Chi-Quadrat | `chi2_contingency` | Zusammenhang von 2 Kategorien |
| Korrelation | `pearsonr` | Zusammenhang von 2 Zahlen |

**Kernaussage:** Der Code ist der einfache Teil. Die Kunst des Analysten besteht darin:
1. den **richtigen** Test für den Datentyp zu wählen;
2. den p-Wert richtig zu **lesen**;
3. den **Kontext** im Blick zu behalten (Effektgröße, Kausalität, praktische Relevanz).

Jetzt sind Sie dran zu üben! 🚀

